# Exploratory Data Analysis (EDA) — Mutual Fund Analytics
**Bluestock Fintech Capstone Project**
Author: Akash Kumar Pandit


## 1. Imports and Database Connection


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import glob
import os

sns.set_theme(style="darkgrid")
conn = sqlite3.connect('../data/processed/database.db')
print("Connected to SQLite database.")


## 2. Load Datasets & Check Summary Info


In [ ]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'].tolist()
print("Tables in Database:", tables)

fund_master = pd.read_sql("SELECT * FROM fund_master", conn)
print("Fund Master Shape:", fund_master.shape)
fund_master.head()


## 3. Fund Master Exploration


In [ ]:
print("Unique Fund Houses:", fund_master['fund_house'].nunique())
print("Unique Categories:", fund_master['category'].unique())
print("Unique Sub-Categories:", fund_master['sub_category'].unique())

plt.figure(figsize=(10, 5))
sns.countplot(data=fund_master, y='sub_category', order=fund_master['sub_category'].value_counts().index, palette='crest')
plt.title("Scheme Count by Sub-Category")
plt.xlabel("Count")
plt.ylabel("Sub-Category")
plt.show()


## 4. Investor Transactions Analysis


In [ ]:
inv_txn = pd.read_sql("SELECT * FROM investor_transactions", conn)
print("Transaction Records:", len(inv_txn))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
inv_txn['transaction_type'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#4f6df5', '#00e5ff', '#ff9800'])
plt.title("Transaction Type Distribution")

plt.subplot(1, 2, 2)
sns.barplot(data=inv_txn.groupby('state')['amount_inr'].sum().reset_index().sort_values('amount_inr', ascending=False).head(10),
            x='amount_inr', y='state', palette='Blues_r')
plt.title("Top 10 States by Transaction Value (INR)")
plt.tight_layout()
plt.show()


## 5. Data Quality & AMFI Code Validation


In [ ]:
nav_codes = set(pd.read_sql("SELECT DISTINCT amfi_code FROM nav_history", conn)['amfi_code'])
master_codes = set(fund_master['amfi_code'])

missing_codes = master_codes - nav_codes
print("Total Codes in Fund Master:", len(master_codes))
print("Total Codes in NAV History:", len(nav_codes))
print("Validation Status:", "PASSED (All codes match)" if len(missing_codes) == 0 else f"FAILED ({len(missing_codes)} missing)")
